<a href="https://colab.research.google.com/github/SonOfGraceProsper/Fake-News-Misinformation-Detector/blob/main/FakeNews_MisinformationDetector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("Starting pipeline...")

# 1. Download a stable public Fake News Dataset
url = "https://raw.githubusercontent.com/docketrun/Detecting-Fake-News-with-Scikit-Learn/master/fake_or_real_news.csv"
csv_path = "fake_or_real_news.csv"

if not os.path.exists(csv_path):
    print("Downloading dataset (this text file is around 30MB, it will take a few seconds)...")
    # Using a user-agent header to bypass potential basic bot blockers
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-agent', 'Mozilla/5.0')]
    urllib.request.install_opener(opener)

    urllib.request.urlretrieve(url, csv_path)
    print("Dataset downloaded successfully!")

# 2. Load and Prepare the Data
df = pd.read_csv(csv_path)
df = df[['title', 'text', 'label']] # Keep relevant columns
df['content'] = df['title'] + " " + df['text'] # Combine title and text body

# Split into inputs and target (1 for FAKE, 0 for REAL)
X = df['content'].fillna('')
y = df['label'].apply(lambda x: 1 if x == 'FAKE' else 0)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Text Vectorization
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7, max_features=5000)
X_train_vectorized = vectorizer.fit_transform(X_train)
X_val_vectorized = vectorizer.transform(X_val)

# 4. Model Training
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vectorized, y_train)

# 5. Evaluation
val_preds = model.predict(X_val_vectorized)
acc = accuracy_score(y_val, val_preds)
print(f"\nModel Training Complete!")
print(f"Validation Accuracy: {acc * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_val, val_preds, target_names=['REAL', 'FAKE']))

# Save files locally for the app later
joblib.dump(model, 'fake_news_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
print("Model and vectorizer saved successfully!")

Starting pipeline...

Model Training Complete!
Validation Accuracy: 91.32%

Classification Report:
              precision    recall  f1-score   support

        REAL       0.92      0.91      0.91       639
        FAKE       0.91      0.92      0.91       628

    accuracy                           0.91      1267
   macro avg       0.91      0.91      0.91      1267
weighted avg       0.91      0.91      0.91      1267

Model and vectorizer saved successfully!


In [4]:
!pip install gradio

import gradio as gr
import joblib

# Load the saved model assets
loaded_model = joblib.load('fake_news_model.pkl')
loaded_vectorizer = joblib.load('vectorizer.pkl')

def detect_misinformation(news_text):
    if not news_text.strip():
        return "Please paste some news text to analyze."

    # Vectorize and predict probability
    vectorized_input = loaded_vectorizer.transform([news_text])
    probabilities = loaded_model.predict_proba(vectorized_input)[0]

    # Label mapping (0: REAL, 1: FAKE)
    results = {
        "Reliable / Real News": float(probabilities[0]),
        "Unreliable / Potential Misinformation": float(probabilities[1])
    }
    return results

# Build custom UI Layout
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as demo:
    gr.Markdown("# 📰 AI Misinformation & Fake News Detector")
    gr.Markdown("Paste the headline or full body text of a news article below to scan it for malicious or suspicious writing patterns.")

    with gr.Row():
        with gr.Column():
            text_input = gr.Textbox(lines=8, placeholder="Paste news article content here...", label="News Content")
            submit_btn = gr.Button("Analyze Text", variant="primary")
        with gr.Column():
            label_output = gr.Label(num_top_classes=2, label="Analysis Verdict")

    submit_btn.click(fn=detect_misinformation, inputs=text_input, outputs=label_output)

demo.launch(share=True)

/tmp/ipykernel_551/3185727949.py:26: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9caec64da566dbcea4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
